# maniol-elvish シーン別 Classifier 推論・自動仕分け（`scene-inspection-002`）

[`scene-classifier-train-001.ipynb`](./scene-classifier-train-001.ipynb) で学習したシーン別分類器を読み込み、`ComfyUI/output/maniol-elvish/` の生成画像を自動でシーン別フォルダへ仕分けるノートブックです（`rtmlib` 不使用）。

- **モデル読込元**: `Colab Notebooks/snapshot-inspector/maniol-elvish/inspection_002/model/`
- **評価・推論対象**: `MyDrive/ComfyUI/output/maniol-elvish/`（サブフォルダ単体指定・全サブフォルダ一括処理どちらも対応）
- **分類後出力先**: `Colab Notebooks/snapshot-inspector/maniol-elvish/inspection_002/classified/`（CSV レポート + ZIP 保存）

1. **セットアップ**: 必要パッケージ導入と GPU 確認
2. **Drive マウント & 学習済みモデル・特徴抽出器初期化**
3. **評価データ確認（`ComfyUI/output/maniol-elvish`）**: サブフォルダ一覧表示と対象選択
4. **推論・信頼度ルーティング設定 & 実行**: ベースシーン + `sitting`/`twisted` 合成、低信頼度は `_uncertain/` へ退避
5. **結果レビュー・可視化**: 分布グラフ、`_uncertain/` のサムネイル確認、任意シーンのサムネイル確認
6. **ローカル書き出し & Google Drive（Colab Notebooks 内）保存**

## 1. セットアップ

In [ ]:
%pip install -q opencv-python huggingface_hub transformers pillow pandas matplotlib scikit-learn joblib
%pip uninstall -y onnxruntime onnxruntime-gpu
%pip install -q --no-cache-dir onnxruntime-gpu

In [ ]:
!nvidia-smi

## 2. Google Drive マウント & 学習済みモデル・特徴抽出器初期化

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/belre/stable-diffusion-pipelines.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/stable-diffusion-pipelines")

local_inspector_root = Path("..").resolve()
if (local_inspector_root / "common" / "__init__.py").exists():
    INSPECTOR_ROOT = local_inspector_root
else:
    if not REPO_DIR.exists():
        !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
    INSPECTOR_ROOT = REPO_DIR / "colab" / "snapshot-inspector"

if str(INSPECTOR_ROOT) not in sys.path:
    sys.path.insert(0, str(INSPECTOR_ROOT))

import copy
import os
import pandas as pd

from common import (
    DEFAULT_DINOV2_REPO,
    DEFAULT_SCENE_FILTER_RULES,
    DEFAULT_SCENE_ROUTING_CONFIG,
    DEFAULT_WD14_REPO,
    DINOv2Embedder,
    WD14Tagger,
    archive_folder_to_drive,
    discover_input_images,
    export_scene_predictions,
    get_onnx_device_config,
    load_scene_classifier_bundle,
    predict_scene_batch,
    save_results_to_drive,
    show_scene_distribution_chart,
    show_scene_thumbnails,
)

CHARACTER_NAME = "maniol-elvish"
DRIVE_INSPECTOR_DIR = f"/content/drive/MyDrive/Colab Notebooks/snapshot-inspector/{CHARACTER_NAME}/inspection_002"
DRIVE_MODEL_DIR = f"{DRIVE_INSPECTOR_DIR}/model"

bundle = load_scene_classifier_bundle(DRIVE_MODEL_DIR)

_, ort_providers = get_onnx_device_config()
dinov2_repo = bundle.get("dinov2_model_id") or DEFAULT_DINOV2_REPO
wd14_repo = bundle.get("wd14_repo_id") or DEFAULT_WD14_REPO

dinov2_embedder = DINOv2Embedder(model_id=dinov2_repo)
wd14_tagger = WD14Tagger(repo_id=wd14_repo, providers=ort_providers)

## 3. 評価データ確認（`ComfyUI/output/maniol-elvish`）

`/content/drive/MyDrive/ComfyUI/output/maniol-elvish` 配下のサブフォルダを確認し、推論対象を選択します。
- `TARGET_SUBFOLDER = None` にすると配下の全サブフォルダを一括推論します。
- 特定バッチのみ実行したい場合は `TARGET_SUBFOLDER = "20261125_224144"` のように指定してください。

In [ ]:
COMFYUI_EVAL_ROOT = f"/content/drive/MyDrive/ComfyUI/output/{CHARACTER_NAME}"
eval_root_path = Path(COMFYUI_EVAL_ROOT)
assert eval_root_path.exists(), f"評価データフォルダが見つかりません: {COMFYUI_EVAL_ROOT}"

subfolders = sorted([p.name for p in eval_root_path.iterdir() if p.is_dir()])
print(f"ComfyUI 評価フォルダ: {COMFYUI_EVAL_ROOT}")
print(f"検出されたサブフォルダ ({len(subfolders)}): {subfolders}")

# None なら全サブフォルダ対象、特定フォルダのみならフォルダ名を指定（例: "20261125_224144"）
TARGET_SUBFOLDER = None

EVAL_INPUT_DIR = (
    str(eval_root_path / TARGET_SUBFOLDER) if TARGET_SUBFOLDER else str(eval_root_path)
)
RUN_LABEL = TARGET_SUBFOLDER or "all_batches"

eval_items = discover_input_images(EVAL_INPUT_DIR)
print(f"推論対象: {EVAL_INPUT_DIR} -> {len(eval_items)} 枚")

## 4. 推論・信頼度ルーティング設定 & 実行

- `min_confidence` / `min_margin` 未満の画像は `_uncertain/` に退避されます。
- WD14 禁止タグフィルタ（`enable_tag_filter`）と DINOv2 葉内重複除外（`enable_dedup`）は必要に応じて `True` / `False` で切り替えられます。

In [ ]:
ROUTING_CONFIG = copy.deepcopy(DEFAULT_SCENE_ROUTING_CONFIG)
ROUTING_CONFIG.update({
    "min_confidence": 0.45,
    "min_margin": 0.05,
    "sitting_threshold": 0.50,
    "twisted_threshold": 0.70,
    "exclusive_modifier": True,
})

FILTER_RULES = copy.deepcopy(DEFAULT_SCENE_FILTER_RULES)
FILTER_RULES.update({
    "enable_tag_filter": False,   # True にすると 1girl 未満や monochrome/multiple_girls を _rejected へ退避
    "enable_dedup": False,        # True にすると葉内の超高類似画像 (>=0.96) を _duplicate へ退避
    "max_leaf_pair_sim_dedup": 0.96,
})

DRIVE_INFER_CACHE_DIR = f"{DRIVE_INSPECTOR_DIR}/cache/infer_{RUN_LABEL}"

pred_df, summary_df = predict_scene_batch(
    image_items=eval_items,
    bundle=bundle,
    embedder=dinov2_embedder,
    tagger=wd14_tagger,
    cache_dir=DRIVE_INFER_CACHE_DIR,
    routing_config=ROUTING_CONFIG,
    filter_rules=FILTER_RULES,
    use_hflip_tta=True,  # v2 モデル時に左右反転 TTA で左右判定を安定化
)

print("\n=== ルート別件数 ===")
display(pred_df["route"].value_counts().to_frame("count"))
print("\n=== シーン別サマリー（上位 30） ===")
display(summary_df.head(30))

## 5. 結果レビュー・可視化

In [ ]:
show_scene_distribution_chart(summary_df, top_n=30)

# _uncertain（低信頼度）のサンプル確認
n_uncertain = int((pred_df["route"] == "uncertain").sum())
print(f"\n_uncertain 件数: {n_uncertain} 枚")
if n_uncertain > 0:
    show_scene_thumbnails(pred_df, route="uncertain", n=12, cols=4, sort_by="confidence_desc")

In [ ]:
# 気になるシーンのサムネイルを確認（低信頼度順に表示すると誤分類チェックがしやすい）
classified_summary = summary_df[summary_df["route"] == "classified"]
if len(classified_summary) > 0:
    INSPECT_LEAF = classified_summary.iloc[0]["final_leaf"]
    print(f"Inspecting classified leaf: {INSPECT_LEAF}")
    show_scene_thumbnails(pred_df, leaf=INSPECT_LEAF, route="classified", n=12, cols=4, sort_by="confidence_asc")

## 6. 書き出し & Google Drive（Colab Notebooks 内）保存

分類後のフォルダ（`classified/<向き>/<フレーミング>/<アングル>/`, `_uncertain/`）をローカルに書き出し、
Google Drive の `Colab Notebooks/snapshot-inspector/maniol-elvish/inspection_002/classified/` へ CSV と ZIP を保存します。

In [ ]:
LOCAL_OUTPUT_DIR = f"./output/{CHARACTER_NAME}/inspection_002/{RUN_LABEL}"
DRIVE_OUTPUT_DIR = f"{DRIVE_INSPECTOR_DIR}/classified/{RUN_LABEL}"

exported = export_scene_predictions(
    pred_df=pred_df,
    output_dir=LOCAL_OUTPUT_DIR,
    clear_export_dirs=True,
    prefix_parent_dir=True,  # 複数日時フォルダの同名ファイル衝突を防止
)

# CSV レポート保存
save_results_to_drive(
    DRIVE_OUTPUT_DIR,
    file_paths=[
        exported["predictions_csv"],
        exported["summary_csv"],
    ],
)

# 分類済みフォルダを ZIP 化して Colab Notebooks 内へ保存
archive_folder_to_drive(
    src_dir=exported["classified_dir"],
    drive_output_dir=DRIVE_OUTPUT_DIR,
    zip_name=f"{CHARACTER_NAME}_inspection002_{RUN_LABEL}_classified.zip",
    local_work_dir=LOCAL_OUTPUT_DIR,
    compress=False,
)

# _uncertain フォルダも存在すれば ZIP 保存
if os.path.exists(exported["uncertain_dir"]) and len(os.listdir(exported["uncertain_dir"])) > 0:
    archive_folder_to_drive(
        src_dir=exported["uncertain_dir"],
        drive_output_dir=DRIVE_OUTPUT_DIR,
        zip_name=f"{CHARACTER_NAME}_inspection002_{RUN_LABEL}_uncertain.zip",
        local_work_dir=LOCAL_OUTPUT_DIR,
        compress=False,
    )

### VS Code 上でのランタイム終了手順
作業が終わったら、GPUランタイムの課金・リソース消費を防ぐため、必ずランタイムを切断・削除してください。

1. VS Code 上部のコマンドパレット（`Ctrl + Shift + P` / macOS は `Cmd + Shift + P`）を開く
2. **`Colab: Remove Server`** を入力して実行する
3. 確認ダイアログが出たら対象の Colab ランタイムを削除して終了する